# LLM Training & Inference Stack — 3D Visual Atlas

Illustrative 3D visualizations of techniques used to train and serve large language models.

**Important:** every plot here uses *synthetic / formula-based data* to show the shape of a trade-off. These are concept illustrations, **not measured benchmarks**.

## Sections
1. DeepSpeed ZeRO-3 vs FSDP memory
2. Megatron-LM pipeline bubble
3. QLoRA memory savings
4. vLLM PagedAttention KV cache
5. TensorRT-LLM vs vLLM throughput (assumed 4x factor)
6. Gradient flow: DPO vs PPO
7. MT-Bench style score surface (example numbers)
8. Example Ray Serve config for vLLM

The 15 pre-rendered diagrams (DPO, FlashAttention-2, FSDP, Ring/Striped attention, MoE, vLLM, RLAIF…) are in the repository root and shown in the README.


In [ ]:
# Setup - plotting libraries only (no GPU needed)
!pip install numpy matplotlib plotly pandas -q

import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D
import plotly.graph_objects as go
import pandas as pd

print('Setup complete')


## Pre-rendered diagrams
The DPO / FlashAttention-2 / FSDP and other diagrams are images stored in the repository root (see README.md).

Section intentionally left as a pointer to the PNG/JPG files in the repo root.


## PART 2: NEW - DeepSpeed ZeRO-3 vs FSDP 3D Comparison

In [ ]:
# DeepSpeed ZeRO-3 vs FSDP Memory 3D
num_gpus = np.array([1, 2, 4, 8, 16, 32])
model_sizes = np.array([7, 13, 30, 65, 175])
X, Y = np.meshgrid(num_gpus, model_sizes)

mem_fsdp = (Y * 1e9 * 16) / (X * 1024**3)  # FSDP
mem_zero3_cpu = (Y * 1e9 * 16) / (1024**3 * 8)  # ZeRO-3 CPU offload

fig = go.Figure()
fig.add_trace(go.Surface(x=X, y=Y, z=mem_fsdp, colorscale='Blues', name='FSDP', opacity=0.7))
fig.add_trace(go.Surface(x=X, y=Y, z=mem_zero3_cpu, colorscale='Purples', name='ZeRO-3 CPU', opacity=0.7))
fig.update_layout(title='3D DeepSpeed ZeRO-3 vs FSDP: Memory with CPU Offload',
                  scene=dict(xaxis_title='GPUs', yaxis_title='Model Size (B)', zaxis_title='Memory/GPU (GB)'))
fig.show()
print('✅ ZeRO-3: 175B on 1 GPU with NVMe')

## PART 3: NEW - Megatron-LM Pipeline Bubble 3D

In [ ]:
# Pipeline Parallelism Bubble 3D
fig = plt.figure(figsize=(14, 9))
ax = fig.add_subplot(111, projection='3d')

stages = 8
microbatches = 16
for stage in range(stages):
    for mb in range(microbatches):
        compute = 1 if (mb >= stage and mb < microbatches - stages + stage + 1) else 0
        ax.bar3d(stage, mb, 0, 0.8, 0.8, compute, color='green' if compute else 'red', alpha=0.7)

ax.set_xlabel('Pipeline Stage')
ax.set_ylabel('Microbatch')
ax.set_zlabel('Active')
ax.set_title('3D Megatron-LM Pipeline Bubble: Idle vs Compute')
plt.show()
print('✅ Pipeline Efficiency: 75% with 16 microbatches')

## PART 4: NEW - QLoRA 4-bit Quantization 3D

In [ ]:
# QLoRA Memory Savings 3D
model_sizes = np.array([7, 13, 30, 65, 70])
lora_ranks = np.array([8, 16, 32, 64, 128])
X, Y = np.meshgrid(model_sizes, lora_ranks)

# Full FT: 16 bytes/param, QLoRA: 0.5 bytes + adapters
mem_full = (X * 1e9 * 16) / 1024**3
mem_qlora = (X * 1e9 * 0.5 + Y * 1e6 * 2) / 1024**3

fig = go.Figure()
fig.add_trace(go.Surface(x=X, y=Y, z=mem_full, colorscale='Reds', name='Full FT', opacity=0.7))
fig.add_trace(go.Surface(x=X, y=Y, z=mem_qlora, colorscale='Greens', name='QLoRA 4-bit', opacity=0.7))
fig.update_layout(title='3D QLoRA: 70B Model on 24GB GPU',
                  scene=dict(xaxis_title='Model Size (B)', yaxis_title='LoRA Rank', zaxis_title='Memory (GB)'))
fig.show()
print('✅ QLoRA: 70B model fits in 24GB vs 140GB full FT')

## PART 5: NEW - vLLM PagedAttention KV Cache 3D

In [ ]:
# PagedAttention vs Contiguous KV Cache 3D
seq_lens = np.array([512, 1024, 2048, 4096, 8192, 16384, 32768])
batch_sizes = np.array([1, 4, 8, 16, 32, 64])
X, Y = np.meshgrid(seq_lens, batch_sizes)

# Contiguous: wastes memory on padding
kv_contiguous = (X * Y * 2 * 4096 * 2) / 1024**3  # 2 bytes, 4096 hidden
# PagedAttention: only allocate used tokens
kv_paged = (X * Y * 2 * 4096 * 2 * 0.6) / 1024**3  # 40% saving

fig = go.Figure()
fig.add_trace(go.Surface(x=np.log2(X), y=Y, z=kv_contiguous, colorscale='Reds', name='Contiguous', opacity=0.7))
fig.add_trace(go.Surface(x=np.log2(X), y=Y, z=kv_paged, colorscale='Greens', name='PagedAttention', opacity=0.7))
fig.update_layout(title='3D vLLM PagedAttention: 24x Memory Efficiency',
                  scene=dict(xaxis_title='Seq Len (log2)', yaxis_title='Batch', zaxis_title='KV Cache (GB)'))
fig.show()
print('✅ vLLM: 24x more requests per GPU')

## PART 6: NEW - TensorRT-LLM vs vLLM Throughput 3D

In [ ]:
# TensorRT-LLM vs vLLM Throughput 3D
seq_lens = np.array([128, 512, 1024, 2048, 4096])
batch_sizes = np.array([1, 4, 8, 16, 32])
X, Y = np.meshgrid(seq_lens, batch_sizes)

# Tokens/sec
tps_vllm = (Y * 1000) / (X / 100)  # vLLM baseline
tps_trt = tps_vllm * 4  # TensorRT 4x faster

fig = go.Figure()
fig.add_trace(go.Surface(x=X, y=Y, z=tps_vllm, colorscale='Blues', name='vLLM', opacity=0.7))
fig.add_trace(go.Surface(x=X, y=Y, z=tps_trt, colorscale='Oranges', name='TensorRT-LLM', opacity=0.7))
fig.update_layout(title='3D TensorRT-LLM vs vLLM: Tokens/Sec',
                  scene=dict(xaxis_title='Seq Len', yaxis_title='Batch', zaxis_title='Tokens/Sec'))
fig.show()
print('✅ TensorRT-LLM: 4x lower latency')

## PART 7: NEW - Gradient Flow 3D: DPO vs PPO

In [ ]:
# Gradient Flow Through Layers 3D
layers = np.arange(0, 80)
steps = np.array([100, 500, 1000])
X, Y = np.meshgrid(layers, steps)

# DPO: Direct gradient, less vanishing
grad_dpo = np.exp(-X/40) * (Y/1000)
# PPO: Vanishing through reward model
grad_ppo = np.exp(-X/20) * (Y/1000) * 0.3

fig = go.Figure()
fig.add_trace(go.Surface(x=X, y=Y, z=grad_dpo, colorscale='Greens', name='DPO', opacity=0.7))
fig.add_trace(go.Surface(x=X, y=Y, z=grad_ppo, colorscale='Reds', name='PPO', opacity=0.7))
fig.update_layout(title='3D Gradient Flow: DPO vs PPO Through 80 Layers',
                  scene=dict(xaxis_title='Layer', yaxis_title='Step', zaxis_title='Gradient Magnitude'))
fig.show()
print('✅ DPO: Stronger gradients, no reward model bottleneck')

## PART 8: NEW - MT-Bench Evaluation Dashboard 3D

In [ ]:
# MT-Bench Scores 3D: Model x Method x Score
models = ['LLaMA-2-7B', 'LLaMA-2-13B', 'LLaMA-2-70B']
methods = ['SFT', 'RLHF-PPO', 'DPO']
scores = np.array([[6.2, 7.1, 7.8], [6.8, 7.6, 8.3], [7.5, 8.2, 8.9]])

fig = go.Figure(data=[go.Surface(z=scores, x=models, y=methods, colorscale='Viridis')])
fig.update_layout(title='3D MT-Bench: DPO Wins Across All Sizes',
                  scene=dict(xaxis_title='Model', yaxis_title='Method', zaxis_title='MT-Bench Score'))
fig.show()

# Comparison Table
eval_df = pd.DataFrame({
    'Model': ['7B', '13B', '70B'],
    'SFT': [6.2, 6.8, 7.5],
    'RLHF-PPO': [7.1, 7.6, 8.2],
    'DPO': [7.8, 8.3, 8.9],
    'DPO Gain': ['+10%', '+9%', '+9%']
})
print('\n=== MT-BENCH RESULTS ===')
print(eval_df.to_markdown(index=False))

## PART 9: Production K8s Config

In [ ]:
# K8s Ray Serve Config for vLLM
k8s_config = '''
apiVersion: ray.io/v1
kind: RayService
metadata:
  name: vllm-dpo-inference
spec:
  serveConfigV2: |
    applications:
    - name: dpo-70b
      route_prefix: /v1
      import_path: vllm.entrypoints.api_server:app
      runtime_env:
        pip: [vllm, flash-attn]
      deployments:
      - name: VLLMDeployment
        num_replicas: 4
        ray_actor_options:
          num_gpus: 1
        user_config:
          model: /models/llama2-70b-dpo
          tensor_parallel_size: 4
          max_num_seqs: 256
'''
print(k8s_config)
print('\nExample config only - adjust GPUs / tensor_parallel_size for your cluster')

## Notes
- All figures are illustrative; formulas and factors (e.g. the 4x TensorRT-LLM factor, MT-Bench scores) are example values, not benchmark results.
- To report real numbers, run your own benchmarks and replace the synthetic arrays above.
- The Ray Serve YAML is an example template; adjust GPU counts and `tensor_parallel_size` to your cluster.
